In [1]:
#from original notebook:
import requests
from rasterio.plot import show
from rasterio.merge import merge
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes 
from mpl_toolkits.axes_grid1.inset_locator import mark_inset
import numpy as np
from pathlib import Path
import os
from urllib.request import urlretrieve
import earthaccess
from earthaccess import Auth, DataCollections, DataGranules, Store

from datetime import datetime
import rasterio
import numpy as np
import pandas as pd
from loguru import logger
from glob import glob
import json
from tqdm import tqdm

/home/rdaroya_umass_edu/miniconda3/envs/ann-ssc/lib/python3.9/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
out_path = "links/b069-opera.npy" 
with open(out_path, "rb") as f:
    npzfile = np.load(f, allow_pickle=True)
    opera_links = npzfile["opera_links"]
    landsat_links = npzfile["landsat_links"]
    sentinel_links = npzfile["sentinel_links"]
    bbox_opera = npzfile["bboxes"]
print(len(opera_links))
print(len(landsat_links))
print(len(sentinel_links))
print(bbox_opera[69])

20870
16800
27414
[-72 -60 -66 -54]


In [3]:
l_links = []
for l in landsat_links:
    if not l.endswith("B01.tif"):
        continue
    l_links.append(l)

s_links = []
for l in sentinel_links:
    if not l.endswith("B01.tif"):
        continue
    s_links.append(l)


In [5]:
len(l_links), len(s_links)

(1120, 1523)

In [10]:
# For each opera link, find corresponding landsat/sentinel prefix
opera_to_hls_mapping = {}
ctr = 0
for o_link in tqdm(opera_links):
    if not o_link.endswith("B01_WTR.tif"):
        continue
    ctr += 1
    opera_sat = o_link.split("_")[-5]
    opera_tile_id = o_link.split("_")[-8]
    opera_datetime = o_link.split("_")[-7]
    opera_year = opera_datetime[:4]
    opera_date = opera_datetime.split("T")[0]
    opera_time = opera_datetime.split("T")[1].strip("Z")
    datetime_object = datetime.strptime(opera_date, '%Y%m%d')
    opera_day_of_year = datetime_object.timetuple().tm_yday
    
    hls_datetime = f"{opera_year}{opera_day_of_year:03d}T{opera_time}"

    if opera_sat.startswith("L"): # landsat
        sat_links = l_links
    else:
        sat_links = s_links
    
    hls_subset = f".{opera_tile_id}.{hls_datetime}"
    for hls_link in sat_links:
        if hls_subset in hls_link:
            opera_to_hls_mapping[o_link] = hls_link
            # print(hls_subset, hls_link)
            break
    

  0%|          | 0/20870 [00:00<?, ?it/s]

100%|██████████| 20870/20870 [00:00<00:00, 29200.54it/s]


In [11]:
len(opera_to_hls_mapping), ctr

(2023, 2087)

In [23]:
o_link

'https://archive.podaac.earthdata.nasa.gov/podaac-ops-cumulus-protected/OPERA_L3_DSWX-HLS_PROVISIONAL_V1/OPERA_L3_DSWx-HLS_T46SBC_20230404T043701Z_20230407T140723Z_S2A_30_v1.0_B01_WTR.tif'